In [ ]:
# ------------------------------------------------------------------
# Libraries
# ------------------------------------------------------------------
import numpy as np
from scipy.stats import truncnorm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from pathlib import Path
import json
import csv
from sklearn.model_selection import StratifiedShuffleSplit


# LdaBoost (Tuned to takle the overfitting)

In [ ]:
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

class LdaBoost:
    def __init__(self,
                 n_estimators=100,
                 learning_rate=0.05,
                 max_depth=3,
                 random_state=None,
                 # --- new options to reduce overfitting ---
                 min_samples_leaf=20,
                 min_samples_split=2,
                 max_leaf_nodes=None,
                 ccp_alpha=0.0,
                 subsample=1.0,                 # 0<subsample<=1 (e.g., 0.7)
                 lda_solver='lsqr',             # 'lsqr' enables shrinkage
                 lda_shrinkage='auto',          # 'auto' = Ledoit-Wolf
                 lda_components=None):          # optional (<= n_classes-1)
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.random_state = random_state

        self.min_samples_leaf = min_samples_leaf
        self.min_samples_split = min_samples_split
        self.max_leaf_nodes = max_leaf_nodes
        self.ccp_alpha = ccp_alpha
        self.subsample = subsample
        self.lda_solver = lda_solver
        self.lda_shrinkage = lda_shrinkage
        self.lda_components = lda_components

        self.rng = np.random.RandomState(random_state) if random_state is not None else np.random

        self.estimators = []
        self.lda_transforms = []
        self.initial_logit = None
        self.classes_ = None

    def softmax(self, F):
        expF = np.exp(F - np.max(F, axis=1, keepdims=True))
        return expF / np.sum(expF, axis=1, keepdims=True)

    def _make_lda(self):
        solver = self.lda_solver
        # if it's 'lsqr', switch to 'eigen' because we need .transform
        if solver == 'lsqr':
            solver = 'eigen'
        kwargs = dict(solver=solver, n_components=self.lda_components)
        if solver != 'svd':                 # shrinkage is supported for 'eigen'
            kwargs['shrinkage'] = self.lda_shrinkage
        return LinearDiscriminantAnalysis(**kwargs)

    def fit(self, X, y):
        n_samples = X.shape[0]
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)
        one_hot_y = np.eye(n_classes)[np.searchsorted(self.classes_, y)]

        prior = np.mean(one_hot_y, axis=0)
        prior = np.clip(prior, 1e-5, 1-1e-5)
        self.initial_logit = np.log(prior)

        F = np.tile(self.initial_logit, (n_samples, 1))

        for m in range(self.n_estimators):
            if m == 0:
                lda = self._make_lda()
                X_lda = lda.fit_transform(X, y)
            else:
                p = self.softmax(F)
                residuals_tmp = one_hot_y - p
                labels = np.argmax(residuals_tmp, axis=1)
                lda = self._make_lda()
                X_lda = lda.fit_transform(X, labels)

            self.lda_transforms.append(lda)

            # residuals with respect to the current state
            p = self.softmax(F)
            residuals = one_hot_y - p  # (n_samples, n_classes)

            # subsampling for stochastic boosting (same subset for all classes)
            if 0 < self.subsample < 1.0:
                n_sub = max(2, int(np.ceil(self.subsample * n_samples)))
                idx_sub = self.rng.choice(n_samples, size=n_sub, replace=False)
                X_fit = X_lda[idx_sub]
            else:
                idx_sub = None
                X_fit = X_lda

            estimators_m = []
            for k in range(n_classes):
                y_fit = residuals[idx_sub, k] if idx_sub is not None else residuals[:, k]
                seed = self.rng.randint(0, 10_000)
                reg = DecisionTreeRegressor(
                    max_depth=self.max_depth,
                    min_samples_leaf=self.min_samples_leaf,
                    min_samples_split=self.min_samples_split,
                    max_leaf_nodes=self.max_leaf_nodes,
                    ccp_alpha=self.ccp_alpha,
                    random_state=seed
                )
                reg.fit(X_fit, y_fit)
                estimators_m.append(reg)

                # update F on ALL samples (not only on the subsample)
                F[:, k] += self.learning_rate * reg.predict(X_lda)

            self.estimators.append(estimators_m)

        return self

    def predict_proba(self, X):
        n_samples = X.shape[0]
        F = np.tile(self.initial_logit, (n_samples, 1))
        for lda, estimators_m in zip(self.lda_transforms, self.estimators):
            X_lda = lda.transform(X)
            for k, reg in enumerate(estimators_m):
                F[:, k] += self.learning_rate * reg.predict(X_lda)
        return self.softmax(F)

    def predict(self, X):
        proba = self.predict_proba(X)
        class_indices = np.argmax(proba, axis=1)
        return self.classes_[class_indices]

    def cross_validate(self, X, y, cv=10):
        skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=self.random_state)
        accuracies = []
        for train_idx, test_idx in skf.split(X, y):
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]
            model = LdaBoost(
                n_estimators=self.n_estimators,
                learning_rate=self.learning_rate,
                max_depth=self.max_depth,
                random_state=self.random_state,
                min_samples_leaf=self.min_samples_leaf,
                min_samples_split=self.min_samples_split,
                max_leaf_nodes=self.max_leaf_nodes,
                ccp_alpha=self.ccp_alpha,
                subsample=self.subsample,
                lda_solver=self.lda_solver,
                lda_shrinkage=self.lda_shrinkage,
                lda_components=self.lda_components
            )
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            accuracies.append(accuracy_score(y_test, y_pred))
        return accuracies


In [4]:
# ============================================================
# Generazione dati gaussiani + mini-tuning Optuna per LdaBoost
# ============================================================
import json
from pathlib import Path

import numpy as np
from scipy.stats import truncnorm, norm

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import make_pipeline

# --- importa LdaBoost ---
try:
    # Se hai un modulo locale:
    # from LdaBoosting import LdaBoost
    LdaBoost  # verifica se è già definita in sessione
except NameError:
    from LdaBoosting import LdaBoost  # fallback: importa dal tuo package

# --- Optuna ---
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner


# -------------------------------
# 1) Utilità per la generazione Σ
# -------------------------------
def make_corr_well_conditioned(M: np.ndarray, min_eig: float = 1e-2, kappa_max: float = 1e3) -> np.ndarray:
    """
    Rende M una matrice di *correlazione* ben condizionata:
    - simmetrizza e impone diag=1
    - shrink verso I con alpha scelto per autovalore minimo >= min_eig
      e numero di condizione <= kappa_max
    """
    A = (M + M.T) / 2.0
    np.fill_diagonal(A, 1.0)

    w = np.linalg.eigvalsh(A)
    lam_min = float(w.min())
    lam_max = float(w.max())

    alpha_psd = 0.0
    if lam_min < min_eig:
        alpha_psd = (min_eig - lam_min) / (1.0 - lam_min)

    if kappa_max is None:
        alpha_kappa = 0.0
    else:
        # cond((1-α)A + αI) = ((1-α)λ_max + α) / ((1-α)λ_min + α) <= kappa_max
        Acoef = 1.0 - lam_max - kappa_max * (1.0 - lam_min)
        Bcoef = kappa_max * lam_min - lam_max
        alpha_kappa = float(np.clip(Bcoef / Acoef, 0.0, 0.999))

    alpha = float(np.clip(max(alpha_psd, alpha_kappa, 0.0), 0.0, 0.999))
    R = (1.0 - alpha) * A + alpha * np.eye(A.shape[0])
    R = (R + R.T) / 2.0
    return R


def generate_cov_matrix(size: int, mean: float, seed: int = None,
                        std_dev: float = 0.05,
                        min_eig: float = 1e-2, kappa_max: float = 1e3) -> np.ndarray:
    """
    Matrice di correlazione simmetrica (diag=1) con off-diagonali ~ N(mean, std_dev^2) troncati in [-1,1],
    poi resa ben condizionata.
    """
    if seed is not None:
        np.random.seed(seed)
    a, b = (-1 - mean) / std_dev, (1 - mean) / std_dev
    M = np.eye(size)
    for i in range(size):
        for j in range(i + 1, size):
            v = truncnorm.rvs(a, b, loc=mean, scale=std_dev)
            M[i, j] = M[j, i] = v
    return make_corr_well_conditioned(M, min_eig=min_eig, kappa_max=kappa_max)


# ----------------------------------------
# 2) Centri di classe e generazione (X, y)
# ----------------------------------------
def class_means(n_features: int, n_classes: int, offset: float) -> np.ndarray:
    means = []
    for k in range(n_classes):
        m = np.zeros(n_features)
        for j in range(n_features):
            if (k >> j) & 1:
                m[j] = offset
        means.append(m)
    return np.asarray(means)


def make_classification_cov(n_samples: int, n_features: int, n_classes: int,
                            cov: np.ndarray, random_state: int = None,
                            offset: float = 1.0, shuffle: bool = True):
    rng = np.random.RandomState(random_state)
    Σ = np.array(cov)
    MU = class_means(n_features, n_classes, offset)

    base, extra = divmod(n_samples, n_classes)
    X_parts, y_parts = [], []
    for cls in range(n_classes):
        n = base + (1 if cls < extra else 0)
        X_parts.append(rng.multivariate_normal(MU[cls], Σ, size=n))
        y_parts.append(np.full(n, cls, dtype=int))
    X = np.vstack(X_parts)
    y = np.concatenate(y_parts)

    if shuffle:
        perm = rng.permutation(n_samples)
        X, y = X[perm], y[perm]
    return X, y


def generate_dataset(n_samples: int, n_features: int, n_classes: int,
                     corr: float, seed: int = 42,
                     offset_signal: float = 1.0,
                     std_dev: float = 0.05,
                     min_eig: float = 1e-2, kappa_max: float = 1e3):
    Σ = generate_cov_matrix(n_features, mean=corr, seed=seed,
                            std_dev=std_dev, min_eig=min_eig, kappa_max=kappa_max)
    X, y = make_classification_cov(n_samples=n_samples, n_features=n_features,
                                   n_classes=n_classes, cov=Σ,
                                   random_state=seed, offset=offset_signal, shuffle=True)
    return X, y, Σ


# --------------------------------------------
# 3) Splits fissi + pipeline + Optuna objective
# --------------------------------------------
def make_fixed_splits(X, y, train_size=2000, seeds=(11, 29)):
    splits = []
    for s in seeds:
        sss = StratifiedShuffleSplit(n_splits=1, train_size=train_size, random_state=s)
        tr_idx, te_idx = next(sss.split(X, y))
        splits.append((tr_idx, te_idx))
    return splits


def build_pipeline(params, k_select, n_classes):
    model = LdaBoost(
        n_estimators=params["n_estimators"],
        learning_rate=params["learning_rate"],
        max_depth=params["max_depth"],
        min_samples_leaf=params["min_samples_leaf"],
        ccp_alpha=params["ccp_alpha"],
        subsample=params["subsample"],
        lda_solver="eigen",        # necessario per transform + shrinkage
        lda_shrinkage="auto",
        lda_components=None,
        random_state=1
    )
    if k_select is None:
        return model
    else:
        return make_pipeline(SelectKBest(f_classif, k=k_select), model)


def objective_factory(X, y, splits, n_classes):
    def objective(trial: optuna.Trial) -> float:
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 75, 250),
            "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.10, log=True),
            "max_depth": trial.suggest_categorical("max_depth", [2, 3]),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 100),
            "ccp_alpha": trial.suggest_float("ccp_alpha", 1e-4, 5e-3, log=True),
            "subsample": trial.suggest_float("subsample", 0.5, 0.9),
        }
        k_select = trial.suggest_categorical("k_select", [None, 20, 50, 100])

        accs = []
        for fold_idx, (tr_idx, te_idx) in enumerate(splits, start=1):
            X_tr, X_te = X[tr_idx], X[te_idx]
            y_tr, y_te = y[tr_idx], y[te_idx]

            pipe = build_pipeline(params, k_select, n_classes)
            pipe.fit(X_tr, y_tr)
            y_pred = pipe.predict(X_te)
            acc = accuracy_score(y_te, y_pred)
            accs.append(acc)

            trial.report(float(np.mean(accs)), step=fold_idx)
            if trial.should_prune():
                raise optuna.TrialPruned()

        return float(np.mean(accs))
    return objective


def run_optuna_tuning_for_config(
    n_samples=2858, n_features=100, n_classes=5, corr=0.3,
    offset_signal=1.0, std_dev=0.05, min_eig=1e-2, kappa_max=1e3,
    train_size=2000, seeds=(11, 29),
    n_trials=30, timeout=None,
    out_dir="optuna_lda_boost_synth"
):
    # 1) Genera dataset
    X, y, Σ = generate_dataset(
        n_samples=n_samples, n_features=n_features, n_classes=n_classes,
        corr=corr, seed=42, offset_signal=offset_signal,
        std_dev=std_dev, min_eig=min_eig, kappa_max=kappa_max
    )

    # Info numeriche su Σ (utile per logs)
    eigvals = np.linalg.eigvalsh(Σ)
    min_eig_Sigma = float(eigvals.min())
    cond_Sigma = float(np.linalg.cond(Σ))

    # 2) Splits fissi
    splits = make_fixed_splits(X, y, train_size=train_size, seeds=seeds)
    n_classes_unique = len(np.unique(y))

    # 3) Optuna
    sampler = TPESampler(seed=42, multivariate=True)
    pruner = MedianPruner(n_warmup_steps=1)

    study = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner,
                                study_name=f"ldaboost_synth_p{n_features}_corr{corr}")
    objective = objective_factory(X, y, splits, n_classes_unique)
    study.optimize(objective, n_trials=n_trials, timeout=timeout, n_jobs=1, show_progress_bar=True)

    # 4) Salvataggi
    out = Path(out_dir) / f"p{n_features}_corr{str(corr).replace('.','_')}"
    out.mkdir(parents=True, exist_ok=True)

    # dataset+Σ info
    with (out / "dataset_info.json").open("w", encoding="utf-8") as f:
        json.dump({
            "n_samples": n_samples,
            "n_features": n_features,
            "n_classes": n_classes,
            "corr": corr,
            "offset_signal": offset_signal,
            "std_dev_offdiag": std_dev,
            "min_eig_floor": min_eig,
            "kappa_max": kappa_max,
            "min_eig_Sigma": min_eig_Sigma,
            "cond_Sigma": cond_Sigma
        }, f, ensure_ascii=False, indent=2)

    # trial dataframe
    df = study.trials_dataframe(attrs=("number", "value", "state", "params", "datetime_start", "datetime_complete"))
    df.to_csv(out / "trials.csv", index=False)

    # best params
    best = {"best_value": study.best_value, "best_params": study.best_params,
            "n_trials": len(study.trials), "seeds": list(seeds), "train_size": train_size}
    with (out / "best.json").open("w", encoding="utf-8") as f:
        json.dump(best, f, ensure_ascii=False, indent=2)

    # 5) Fit finale su uno split per una stima onesta
    tr_idx, te_idx = splits[0]
    X_tr, X_te = X[tr_idx], X[te_idx]
    y_tr, y_te = y[tr_idx], y[te_idx]

    k_select = study.best_params.get("k_select", None)
    final_params = {k: v for k, v in study.best_params.items() if k != "k_select"}
    pipe_best = build_pipeline(final_params, k_select, n_classes_unique)
    pipe_best.fit(X_tr, y_tr)
    acc_train = accuracy_score(y_tr, pipe_best.predict(X_tr))
    acc_test  = accuracy_score(y_te, pipe_best.predict(X_te))
    with (out / "final_metrics.json").open("w", encoding="utf-8") as f:
        json.dump({"acc_train": float(acc_train), "acc_test": float(acc_test)}, f, ensure_ascii=False, indent=2)

    print("\n== Best params ==")
    print(json.dumps(best, indent=2))
    print(f"Final (seed={seeds[0]}): acc_train={acc_train:.3f}  acc_test={acc_test:.3f}")
    print(f"Artefatti in: {out.resolve()}")

    return study, pipe_best, (X, y, Σ)

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


-----

# Five class target

In [39]:
# ------------------------------------------------------------------
# 0. Utility per garantire PSD
# ------------------------------------------------------------------
def make_corr_well_conditioned(M: np.ndarray, min_eig: float = 1e-2, kappa_max: float = 1e3) -> np.ndarray:
    """
    Rende M una matrice di *correlazione* ben condizionata:
    - simmetrizza
    - mantiene diag=1
    - 'shrink' verso I con un alpha scelto per:
        * garantire autovalore minimo >= min_eig
        * garantire numero di condizione <= kappa_max
    Non serve rinormalizzare la diagonale: rimane 1 esattamente.
    """
    A = (M + M.T) / 2.0
    np.fill_diagonal(A, 1.0)

    # Spettro
    w = np.linalg.eigvalsh(A)
    lam_min = float(w.min())
    lam_max = float(w.max())

    # alpha per garantire PSD con autovalore minimo >= min_eig
    alpha_psd = 0.0
    if lam_min < min_eig:
        # (1 - α)*lam_min + α = min_eig  -> α = (min_eig - lam_min) / (1 - lam_min)
        alpha_psd = (min_eig - lam_min) / (1.0 - lam_min)

    # alpha per garantire condizione <= kappa_max
    # cond((1-α)A + αI) = ((1-α)lam_max + α) / ((1-α)lam_min + α)
    if kappa_max is None:
        alpha_kappa = 0.0
    else:
        Acoef = 1.0 - lam_max - kappa_max * (1.0 - lam_min)
        Bcoef = kappa_max * lam_min - lam_max
        # Se già cond <= kappa_max, questo risulta 0 o negativo -> clip a [0, 0.999]
        alpha_kappa = float(np.clip(Bcoef / Acoef, 0.0, 0.999))

    alpha = float(np.clip(max(alpha_psd, alpha_kappa, 0.0), 0.0, 0.999))

    R = (1.0 - alpha) * A + alpha * np.eye(A.shape[0])
    R = (R + R.T) / 2.0  # pulizia numerica
    # Diagonale resta 1 esatta
    return R


# ------------------------------------------------------------------
# 1. Funzioni di utilità -- come in precedenza (con fix PSD)
# ------------------------------------------------------------------
def generate_cov_matrix(size: int, mean: float, seed: int = None,
                        std_dev: float = 0.05) -> np.ndarray:
    """
    Ritorna una matrice *di correlazione* simmetrica con 1 sulla diagonale
    e valori ~ 𝒩(mean, std_dev²) (troncati in [-1,1]) fuori diagonale.
    La matrice è poi proiettata alla più vicina PSD e rinormalizzata.
    """
    if seed is not None:
        np.random.seed(seed)

    a, b = (-1 - mean) / std_dev, (1 - mean) / std_dev
    M = np.eye(size)
    for i in range(size):
        for j in range(i + 1, size):
            v = truncnorm.rvs(a, b, loc=mean, scale=std_dev)
            M[i, j] = M[j, i] = v

    # Proietta a PSD e rinormalizza (diag=1)
    M = make_corr_well_conditioned(M, min_eig=1e-2, kappa_max=1e3)
    return M


def make_classification_cov(n_samples: int, n_features: int, n_classes: int,
                            cov=None, shuffle: bool = True,
                            random_state: int = None):
    """
    Dataset Gaussiano multiclasse con covarianza arbitraria 'cov'.
    """
    rng = np.random.RandomState(random_state)
    Σ = np.eye(n_features) if cov is None else np.array(cov)

    # semplicissima codifica dei centri di classe
    offset = 1.0
    means = []
    for k in range(n_classes):
        m = np.zeros(n_features)
        for j in range(n_features):
            if (k >> j) & 1:
                m[j] = offset
        means.append(m)
    means = np.asarray(means)

    # campionamento
    base, extra = divmod(n_samples, n_classes)
    X_parts, y_parts = [], []
    for cls in range(n_classes):
        n = base + (1 if cls < extra else 0)
        X_parts.append(rng.multivariate_normal(means[cls], Σ, size=n))
        y_parts.append(np.full(n, cls, dtype=int))

    X = np.vstack(X_parts)
    y = np.concatenate(y_parts)

    if shuffle:
        perm = rng.permutation(n_samples)
        X, y = X[perm], y[perm]
    return X, y


# ------------------------------------------------------------------
# 2. Parametri dell’esperimento
# ------------------------------------------------------------------
feature_list   = [5, 50, 100, 300, 400]
correlations   = [0.0, 0.3, 0.7]
n_reps         = 5          # dataset diversi per stimare varianza
n_classes      = 5
n_samples   = 2858
train_size  = 2000

# dizionario: results[corr][n_features] -> lista accuracy (len = n_reps)
results = {c: {f: [] for f in feature_list} for c in correlations}

# Cartella output
out_dir = Path("output_lda_boost")
out_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# 3. Loop principale: generazione, training, valutazione
# ------------------------------------------------------------------
for corr in correlations:
    for n_feat in feature_list:
        for rep in range(1, n_reps + 1):
            print(f"features: {n_feat}, stage: {rep}, corr: {corr}")
            Σ = generate_cov_matrix(n_feat, mean=corr,
                                    seed=rep, std_dev=0.05)

            X, y = make_classification_cov(n_samples=n_samples,
                                           n_features=n_feat,
                                           n_classes=n_classes,
                                           cov=Σ,
                                           random_state=rep)
            sss = StratifiedShuffleSplit(n_splits=1, train_size=train_size, random_state=1)
            train_idx, test_idx = next(sss.split(X, y))
            X_tr, X_te = X[train_idx], X[test_idx]
            y_tr, y_te = y[train_idx], y[test_idx]

            model = LdaBoost(n_estimators=186,
                             learning_rate=0.05577840617948812,
                             max_depth=3,
                             min_samples_leaf=89,
                             ccp_alpha = 0.00015696331582345908,
                             subsample = 0.7651912630747271,
                             random_state=1)
            model.fit(X_tr, y_tr)
            acc = accuracy_score(y_te, model.predict(X_te))

            results[corr][n_feat].append(acc)

# ------------------------------------------------------------------
# 4. Preparazione statistiche (media ± std) e stampa
# ------------------------------------------------------------------
means = {c: [np.mean(results[c][f]) for f in feature_list]
         for c in correlations}
stds  = {c: [np.std (results[c][f]) for f in feature_list]
         for c in correlations}

print("\n=== Risultati (accuracy media ± std) su", n_reps, "repliche ===")
for c in correlations:
    for idx, f in enumerate(feature_list):
        mu = means[c][idx]
        sd = stds[c][idx]
        print(f"corr={c:>3}, n_features={f:>3} -> mean={mu:.4f}, std={sd:.4f}")

# ------------------------------------------------------------------
# 5. Salvataggio su file (CSV + JSON, incluse le rep grezze)
# ------------------------------------------------------------------
# CSV 'summary'
csv_path = out_dir / "summary_means_stds_multiclass.csv"
with csv_path.open("w", newline="") as fh:
    writer = csv.writer(fh)
    writer.writerow(["correlation", "n_features", "n_reps", "mean_acc", "std_acc"])
    for c in correlations:
        for idx, f in enumerate(feature_list):
            writer.writerow([c, f, n_reps, f"{means[c][idx]:.6f}", f"{stds[c][idx]:.6f}"])

# JSON 'summary' strutturato per facile riuso
json_summary_path = out_dir / "summary_means_stds_multiclass.json"
summary_json = {
    "feature_list": feature_list,
    "correlations": correlations,
    "n_reps": n_reps,
    "means": {str(c): means[c] for c in correlations},
    "stds":  {str(c): stds[c]  for c in correlations},
}
with json_summary_path.open("w", encoding="utf-8") as fh:
    json.dump(summary_json, fh, ensure_ascii=False, indent=2)

# JSON con risultati grezzi (per tracciabilità)
json_raw_path = out_dir / "raw_results_multiclass.json"
raw_json = {str(c): {str(f): results[c][f] for f in feature_list} for c in correlations}
with json_raw_path.open("w", encoding="utf-8") as fh:
    json.dump(raw_json, fh, ensure_ascii=False, indent=2)

print(f"\nFile salvati in: {out_dir.resolve()}")
print(f"- {csv_path.name}")
print(f"- {json_summary_path.name}")
print(f"- {json_raw_path.name}")

features: 5, stage: 1, corr: 0.0
features: 5, stage: 2, corr: 0.0
features: 5, stage: 3, corr: 0.0
features: 5, stage: 4, corr: 0.0
features: 5, stage: 5, corr: 0.0
features: 50, stage: 1, corr: 0.0
features: 50, stage: 2, corr: 0.0
features: 50, stage: 3, corr: 0.0
features: 50, stage: 4, corr: 0.0
features: 50, stage: 5, corr: 0.0
features: 100, stage: 1, corr: 0.0
features: 100, stage: 2, corr: 0.0
features: 100, stage: 3, corr: 0.0
features: 100, stage: 4, corr: 0.0
features: 100, stage: 5, corr: 0.0
features: 300, stage: 1, corr: 0.0
features: 300, stage: 2, corr: 0.0
features: 300, stage: 3, corr: 0.0
features: 300, stage: 4, corr: 0.0
features: 300, stage: 5, corr: 0.0
features: 400, stage: 1, corr: 0.0
features: 400, stage: 2, corr: 0.0
features: 400, stage: 3, corr: 0.0
features: 400, stage: 4, corr: 0.0
features: 400, stage: 5, corr: 0.0
features: 5, stage: 1, corr: 0.3
features: 5, stage: 2, corr: 0.3
features: 5, stage: 3, corr: 0.3
features: 5, stage: 4, corr: 0.3
features

# Binary target

In [40]:
# ------------------------------------------------------------------
# 2. Parametri dell’esperimento
# ------------------------------------------------------------------
feature_list   = [5, 50, 100, 300, 400]
correlations   = [0.0, 0.3, 0.7]
n_reps         = 5          # dataset diversi per stimare varianza
n_classes      = 2
n_samples   = 2858
train_size  = 2000

# dizionario: results[corr][n_features] -> lista accuracy (len = n_reps)
results = {c: {f: [] for f in feature_list} for c in correlations}

# Cartella output
out_dir = Path("output_lda_boost")
out_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# 3. Loop principale: generazione, training, valutazione
# ------------------------------------------------------------------
for corr in correlations:
    for n_feat in feature_list:
        for rep in range(1, n_reps + 1):
            print(f"features: {n_feat}, stage: {rep}, corr: {corr}")
            Σ = generate_cov_matrix(n_feat, mean=corr,
                                    seed=rep, std_dev=0.05)

            X, y = make_classification_cov(n_samples=n_samples,
                                           n_features=n_feat,
                                           n_classes=n_classes,
                                           cov=Σ,
                                           random_state=rep)
            sss = StratifiedShuffleSplit(n_splits=1, train_size=train_size, random_state=1)
            train_idx, test_idx = next(sss.split(X, y))
            X_tr, X_te = X[train_idx], X[test_idx]
            y_tr, y_te = y[train_idx], y[test_idx]

            model = LdaBoost(n_estimators=186,
                             learning_rate=0.05577840617948812,
                             max_depth=3,
                             min_samples_leaf=89,
                             ccp_alpha = 0.00015696331582345908,
                             subsample = 0.7651912630747271,
                             random_state=1)
            
            model.fit(X_tr, y_tr)
            acc = accuracy_score(y_te, model.predict(X_te))

            results[corr][n_feat].append(acc)
# ------------------------------------------------------------------
# 4. Preparazione statistiche (media ± std) e stampa
# ------------------------------------------------------------------
means = {c: [np.mean(results[c][f]) for f in feature_list]
         for c in correlations}
stds  = {c: [np.std (results[c][f]) for f in feature_list]
         for c in correlations}

print("\n=== Risultati (accuracy media ± std) su", n_reps, "repliche ===")
for c in correlations:
    for idx, f in enumerate(feature_list):
        mu = means[c][idx]
        sd = stds[c][idx]
        print(f"corr={c:>3}, n_features={f:>3} -> mean={mu:.4f}, std={sd:.4f}")

# ------------------------------------------------------------------
# 5. Salvataggio su file (CSV + JSON, incluse le rep grezze)
# ------------------------------------------------------------------
# CSV 'summary'
csv_path = out_dir / "summary_means_stds_binary.csv"
with csv_path.open("w", newline="") as fh:
    writer = csv.writer(fh)
    writer.writerow(["correlation", "n_features", "n_reps", "mean_acc", "std_acc"])
    for c in correlations:
        for idx, f in enumerate(feature_list):
            writer.writerow([c, f, n_reps, f"{means[c][idx]:.6f}", f"{stds[c][idx]:.6f}"])

# JSON 'summary' strutturato per facile riuso
json_summary_path = out_dir / "summary_means_stds_binary.json"
summary_json = {
    "feature_list": feature_list,
    "correlations": correlations,
    "n_reps": n_reps,
    "means": {str(c): means[c] for c in correlations},
    "stds":  {str(c): stds[c]  for c in correlations},
}
with json_summary_path.open("w", encoding="utf-8") as fh:
    json.dump(summary_json, fh, ensure_ascii=False, indent=2)

# JSON con risultati grezzi (per tracciabilità)
json_raw_path = out_dir / "raw_results_binary.json"
raw_json = {str(c): {str(f): results[c][f] for f in feature_list} for c in correlations}
with json_raw_path.open("w", encoding="utf-8") as fh:
    json.dump(raw_json, fh, ensure_ascii=False, indent=2)

print(f"\nFile salvati in: {out_dir.resolve()}")
print(f"- {csv_path.name}")
print(f"- {json_summary_path.name}")
print(f"- {json_raw_path.name}")

features: 5, stage: 1, corr: 0.0
features: 5, stage: 2, corr: 0.0
features: 5, stage: 3, corr: 0.0
features: 5, stage: 4, corr: 0.0
features: 5, stage: 5, corr: 0.0
features: 50, stage: 1, corr: 0.0
features: 50, stage: 2, corr: 0.0
features: 50, stage: 3, corr: 0.0
features: 50, stage: 4, corr: 0.0
features: 50, stage: 5, corr: 0.0
features: 100, stage: 1, corr: 0.0
features: 100, stage: 2, corr: 0.0
features: 100, stage: 3, corr: 0.0
features: 100, stage: 4, corr: 0.0
features: 100, stage: 5, corr: 0.0
features: 300, stage: 1, corr: 0.0
features: 300, stage: 2, corr: 0.0
features: 300, stage: 3, corr: 0.0
features: 300, stage: 4, corr: 0.0
features: 300, stage: 5, corr: 0.0
features: 400, stage: 1, corr: 0.0
features: 400, stage: 2, corr: 0.0
features: 400, stage: 3, corr: 0.0
features: 400, stage: 4, corr: 0.0
features: 400, stage: 5, corr: 0.0
features: 5, stage: 1, corr: 0.3
features: 5, stage: 2, corr: 0.3
features: 5, stage: 3, corr: 0.3
features: 5, stage: 4, corr: 0.3
features

# Five class target, obs. 10.000

In [41]:
# ------------------------------------------------------------------
# 2. Parametri dell’esperimento
# ------------------------------------------------------------------
feature_list   = [5, 50, 100, 300, 400]
correlations   = [0.0, 0.3, 0.7]
n_reps         = 5          # dataset diversi per stimare varianza
n_classes      = 5
n_samples   = 10000
train_size  = 8000

# dizionario: results[corr][n_features] -> lista accuracy (len = n_reps)
results = {c: {f: [] for f in feature_list} for c in correlations}

# Cartella output
out_dir = Path("output_lda_boost")
out_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# 3. Loop principale: generazione, training, valutazione
# ------------------------------------------------------------------
for corr in correlations:
    for n_feat in feature_list:
        for rep in range(1, n_reps + 1):
            print(f"features: {n_feat}, stage: {rep}, corr: {corr}")
            Σ = generate_cov_matrix(n_feat, mean=corr,
                                    seed=rep, std_dev=0.05)

            X, y = make_classification_cov(n_samples=n_samples,
                                           n_features=n_feat,
                                           n_classes=n_classes,
                                           cov=Σ,
                                           random_state=rep)
            sss = StratifiedShuffleSplit(n_splits=1, train_size=train_size, random_state=1)
            train_idx, test_idx = next(sss.split(X, y))
            X_tr, X_te = X[train_idx], X[test_idx]
            y_tr, y_te = y[train_idx], y[test_idx]

            model = LdaBoost(n_estimators=186,
                             learning_rate=0.05577840617948812,
                             max_depth=3,
                             min_samples_leaf=89,
                             ccp_alpha = 0.00015696331582345908,
                             subsample = 0.7651912630747271,
                             random_state=1)
            model.fit(X_tr, y_tr)
            acc = accuracy_score(y_te, model.predict(X_te))

            results[corr][n_feat].append(acc)
# ------------------------------------------------------------------
# 4. Preparazione statistiche (media ± std) e stampa
# ------------------------------------------------------------------
means = {c: [np.mean(results[c][f]) for f in feature_list]
         for c in correlations}
stds  = {c: [np.std (results[c][f]) for f in feature_list]
         for c in correlations}

print("\n=== Risultati (accuracy media ± std) su", n_reps, "repliche ===")
for c in correlations:
    for idx, f in enumerate(feature_list):
        mu = means[c][idx]
        sd = stds[c][idx]
        print(f"corr={c:>3}, n_features={f:>3} -> mean={mu:.4f}, std={sd:.4f}")

# ------------------------------------------------------------------
# 5. Salvataggio su file (CSV + JSON, incluse le rep grezze)
# ------------------------------------------------------------------
# CSV 'summary'
csv_path = out_dir / "summary_means_stds_multiclass_10k.csv"
with csv_path.open("w", newline="") as fh:
    writer = csv.writer(fh)
    writer.writerow(["correlation", "n_features", "n_reps", "mean_acc", "std_acc"])
    for c in correlations:
        for idx, f in enumerate(feature_list):
            writer.writerow([c, f, n_reps, f"{means[c][idx]:.6f}", f"{stds[c][idx]:.6f}"])

# JSON 'summary' strutturato per facile riuso
json_summary_path = out_dir / "summary_means_stds_multiclass_10k.json"
summary_json = {
    "feature_list": feature_list,
    "correlations": correlations,
    "n_reps": n_reps,
    "means": {str(c): means[c] for c in correlations},
    "stds":  {str(c): stds[c]  for c in correlations},
}
with json_summary_path.open("w", encoding="utf-8") as fh:
    json.dump(summary_json, fh, ensure_ascii=False, indent=2)

# JSON con risultati grezzi (per tracciabilità)
json_raw_path = out_dir / "raw_results_multiclass_10k.json"
raw_json = {str(c): {str(f): results[c][f] for f in feature_list} for c in correlations}
with json_raw_path.open("w", encoding="utf-8") as fh:
    json.dump(raw_json, fh, ensure_ascii=False, indent=2)

print(f"\nFile salvati in: {out_dir.resolve()}")
print(f"- {csv_path.name}")
print(f"- {json_summary_path.name}")
print(f"- {json_raw_path.name}")

features: 5, stage: 1, corr: 0.0
features: 5, stage: 2, corr: 0.0
features: 5, stage: 3, corr: 0.0
features: 5, stage: 4, corr: 0.0
features: 5, stage: 5, corr: 0.0
features: 50, stage: 1, corr: 0.0
features: 50, stage: 2, corr: 0.0
features: 50, stage: 3, corr: 0.0
features: 50, stage: 4, corr: 0.0
features: 50, stage: 5, corr: 0.0
features: 100, stage: 1, corr: 0.0
features: 100, stage: 2, corr: 0.0
features: 100, stage: 3, corr: 0.0
features: 100, stage: 4, corr: 0.0
features: 100, stage: 5, corr: 0.0
features: 300, stage: 1, corr: 0.0
features: 300, stage: 2, corr: 0.0
features: 300, stage: 3, corr: 0.0
features: 300, stage: 4, corr: 0.0
features: 300, stage: 5, corr: 0.0
features: 400, stage: 1, corr: 0.0
features: 400, stage: 2, corr: 0.0
features: 400, stage: 3, corr: 0.0
features: 400, stage: 4, corr: 0.0
features: 400, stage: 5, corr: 0.0
features: 5, stage: 1, corr: 0.3
features: 5, stage: 2, corr: 0.3
features: 5, stage: 3, corr: 0.3
features: 5, stage: 4, corr: 0.3
features

# Binary target, obs. 10.000

In [42]:
# ------------------------------------------------------------------
# 2. Parametri dell’esperimento
# ------------------------------------------------------------------
feature_list   = [5, 50, 100, 300, 400]
correlations   = [0.0, 0.3, 0.7]
n_reps         = 5          # dataset diversi per stimare varianza
n_classes      = 2
n_samples   = 10000
train_size  = 8000

# dizionario: results[corr][n_features] -> lista accuracy (len = n_reps)
results = {c: {f: [] for f in feature_list} for c in correlations}

# Cartella output
out_dir = Path("output_lda_boost")
out_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# 3. Loop principale: generazione, training, valutazione
# ------------------------------------------------------------------
for corr in correlations:
    for n_feat in feature_list:
        for rep in range(1, n_reps + 1):
            print(f"features: {n_feat}, stage: {rep}, corr: {corr}")
            Σ = generate_cov_matrix(n_feat, mean=corr,
                                    seed=rep, std_dev=0.05)

            X, y = make_classification_cov(n_samples=n_samples,
                                           n_features=n_feat,
                                           n_classes=n_classes,
                                           cov=Σ,
                                           random_state=rep)
            sss = StratifiedShuffleSplit(n_splits=1, train_size=train_size, random_state=1)
            train_idx, test_idx = next(sss.split(X, y))
            X_tr, X_te = X[train_idx], X[test_idx]
            y_tr, y_te = y[train_idx], y[test_idx]

            model = LdaBoost(n_estimators=186,
                             learning_rate=0.05577840617948812,
                             max_depth=3,
                             min_samples_leaf=89,
                             ccp_alpha = 0.00015696331582345908,
                             subsample = 0.7651912630747271,
                             random_state=1)
            model.fit(X_tr, y_tr)
            acc = accuracy_score(y_te, model.predict(X_te))

            results[corr][n_feat].append(acc)
# ------------------------------------------------------------------
# 4. Preparazione statistiche (media ± std) e stampa
# ------------------------------------------------------------------
means = {c: [np.mean(results[c][f]) for f in feature_list]
         for c in correlations}
stds  = {c: [np.std (results[c][f]) for f in feature_list]
         for c in correlations}

print("\n=== Risultati (accuracy media ± std) su", n_reps, "repliche ===")
for c in correlations:
    for idx, f in enumerate(feature_list):
        mu = means[c][idx]
        sd = stds[c][idx]
        print(f"corr={c:>3}, n_features={f:>3} -> mean={mu:.4f}, std={sd:.4f}")

# ------------------------------------------------------------------
# 5. Salvataggio su file (CSV + JSON, incluse le rep grezze)
# ------------------------------------------------------------------
# CSV 'summary'
csv_path = out_dir / "summary_means_stds_binary_10k.csv"
with csv_path.open("w", newline="") as fh:
    writer = csv.writer(fh)
    writer.writerow(["correlation", "n_features", "n_reps", "mean_acc", "std_acc"])
    for c in correlations:
        for idx, f in enumerate(feature_list):
            writer.writerow([c, f, n_reps, f"{means[c][idx]:.6f}", f"{stds[c][idx]:.6f}"])

# JSON 'summary' strutturato per facile riuso
json_summary_path = out_dir / "summary_means_stds_binary_10k.json"
summary_json = {
    "feature_list": feature_list,
    "correlations": correlations,
    "n_reps": n_reps,
    "means": {str(c): means[c] for c in correlations},
    "stds":  {str(c): stds[c]  for c in correlations},
}
with json_summary_path.open("w", encoding="utf-8") as fh:
    json.dump(summary_json, fh, ensure_ascii=False, indent=2)

# JSON con risultati grezzi (per tracciabilità)
json_raw_path = out_dir / "raw_results_binary_10k.json"
raw_json = {str(c): {str(f): results[c][f] for f in feature_list} for c in correlations}
with json_raw_path.open("w", encoding="utf-8") as fh:
    json.dump(raw_json, fh, ensure_ascii=False, indent=2)

print(f"\nFile salvati in: {out_dir.resolve()}")
print(f"- {csv_path.name}")
print(f"- {json_summary_path.name}")
print(f"- {json_raw_path.name}")

features: 5, stage: 1, corr: 0.0
features: 5, stage: 2, corr: 0.0
features: 5, stage: 3, corr: 0.0
features: 5, stage: 4, corr: 0.0
features: 5, stage: 5, corr: 0.0
features: 50, stage: 1, corr: 0.0
features: 50, stage: 2, corr: 0.0
features: 50, stage: 3, corr: 0.0
features: 50, stage: 4, corr: 0.0
features: 50, stage: 5, corr: 0.0
features: 100, stage: 1, corr: 0.0
features: 100, stage: 2, corr: 0.0
features: 100, stage: 3, corr: 0.0
features: 100, stage: 4, corr: 0.0
features: 100, stage: 5, corr: 0.0
features: 300, stage: 1, corr: 0.0
features: 300, stage: 2, corr: 0.0
features: 300, stage: 3, corr: 0.0
features: 300, stage: 4, corr: 0.0
features: 300, stage: 5, corr: 0.0
features: 400, stage: 1, corr: 0.0
features: 400, stage: 2, corr: 0.0
features: 400, stage: 3, corr: 0.0
features: 400, stage: 4, corr: 0.0
features: 400, stage: 5, corr: 0.0
features: 5, stage: 1, corr: 0.3
features: 5, stage: 2, corr: 0.3
features: 5, stage: 3, corr: 0.3
features: 5, stage: 4, corr: 0.3
features

------

# Five class target, obs. 1.000 

In [10]:
# ------------------------------------------------------------------
# 2. Parametri dell’esperimento
# ------------------------------------------------------------------
feature_list   = [5, 50, 100, 300, 400]
correlations   = [0.0, 0.3, 0.7]
n_reps         = 5          # dataset diversi per stimare varianza
n_samples      = 1000
n_classes      = 5
train_size      = 0.70

# dizionario: results[corr][n_features] -> lista accuracy (len = n_reps)
results = {c: {f: [] for f in feature_list} for c in correlations}

# Cartella output
out_dir = Path("output_lda_boost")
out_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# 3. Loop principale: generazione, training, valutazione
# ------------------------------------------------------------------
for corr in correlations:
    for n_feat in feature_list:
        for rep in range(1, n_reps + 1):
            print(f"features: {n_feat}, stage: {rep}, corr: {corr}")
            Σ = generate_cov_matrix(n_feat, mean=corr,
                                    seed=rep, std_dev=0.05)

            X, y = make_classification_cov(n_samples=n_samples,
                                           n_features=n_feat,
                                           n_classes=n_classes,
                                           cov=Σ,
                                           random_state=rep)
            sss = StratifiedShuffleSplit(n_splits=1, train_size=train_size, random_state=1)
            train_idx, test_idx = next(sss.split(X, y))
            X_tr, X_te = X[train_idx], X[test_idx]
            y_tr, y_te = y[train_idx], y[test_idx]

            model = LdaBoost(n_estimators=186,
                             learning_rate=0.05577840617948812,
                             max_depth=3,
                             min_samples_leaf=89,
                             ccp_alpha = 0.00015696331582345908,
                             subsample = 0.7651912630747271,
                             random_state=1)
            model.fit(X_tr, y_tr)
            acc = accuracy_score(y_te, model.predict(X_te))

            results[corr][n_feat].append(acc)
# ------------------------------------------------------------------
# 4. Preparazione statistiche (media ± std) e stampa
# ------------------------------------------------------------------
means = {c: [np.mean(results[c][f]) for f in feature_list]
         for c in correlations}
stds  = {c: [np.std (results[c][f]) for f in feature_list]
         for c in correlations}

print("\n=== Risultati (accuracy media ± std) su", n_reps, "repliche ===")
for c in correlations:
    for idx, f in enumerate(feature_list):
        mu = means[c][idx]
        sd = stds[c][idx]
        print(f"corr={c:>3}, n_features={f:>3} -> mean={mu:.4f}, std={sd:.4f}")

# ------------------------------------------------------------------
# 5. Salvataggio su file (CSV + JSON, incluse le rep grezze)
# ------------------------------------------------------------------
# CSV 'summary'
csv_path = out_dir / "summary_means_stds_multiclass_1k.csv"
with csv_path.open("w", newline="") as fh:
    writer = csv.writer(fh)
    writer.writerow(["correlation", "n_features", "n_reps", "mean_acc", "std_acc"])
    for c in correlations:
        for idx, f in enumerate(feature_list):
            writer.writerow([c, f, n_reps, f"{means[c][idx]:.6f}", f"{stds[c][idx]:.6f}"])

# JSON 'summary' strutturato per facile riuso
json_summary_path = out_dir / "summary_means_stds_multiclass_1k.json"
summary_json = {
    "feature_list": feature_list,
    "correlations": correlations,
    "n_reps": n_reps,
    "means": {str(c): means[c] for c in correlations},
    "stds":  {str(c): stds[c]  for c in correlations},
}
with json_summary_path.open("w", encoding="utf-8") as fh:
    json.dump(summary_json, fh, ensure_ascii=False, indent=2)

# JSON con risultati grezzi (per tracciabilità)
json_raw_path = out_dir / "raw_results_multiclass_1k.json"
raw_json = {str(c): {str(f): results[c][f] for f in feature_list} for c in correlations}
with json_raw_path.open("w", encoding="utf-8") as fh:
    json.dump(raw_json, fh, ensure_ascii=False, indent=2)

print(f"\nFile salvati in: {out_dir.resolve()}")
print(f"- {csv_path.name}")
print(f"- {json_summary_path.name}")
print(f"- {json_raw_path.name}")

features: 5, stage: 1, corr: 0.0
features: 5, stage: 2, corr: 0.0
features: 5, stage: 3, corr: 0.0
features: 5, stage: 4, corr: 0.0
features: 5, stage: 5, corr: 0.0
features: 50, stage: 1, corr: 0.0
features: 50, stage: 2, corr: 0.0
features: 50, stage: 3, corr: 0.0
features: 50, stage: 4, corr: 0.0
features: 50, stage: 5, corr: 0.0
features: 100, stage: 1, corr: 0.0
features: 100, stage: 2, corr: 0.0
features: 100, stage: 3, corr: 0.0
features: 100, stage: 4, corr: 0.0
features: 100, stage: 5, corr: 0.0
features: 300, stage: 1, corr: 0.0
features: 300, stage: 2, corr: 0.0
features: 300, stage: 3, corr: 0.0
features: 300, stage: 4, corr: 0.0
features: 300, stage: 5, corr: 0.0
features: 400, stage: 1, corr: 0.0
features: 400, stage: 2, corr: 0.0
features: 400, stage: 3, corr: 0.0
features: 400, stage: 4, corr: 0.0
features: 400, stage: 5, corr: 0.0
features: 5, stage: 1, corr: 0.3
features: 5, stage: 2, corr: 0.3
features: 5, stage: 3, corr: 0.3
features: 5, stage: 4, corr: 0.3
features

# Binary target, obs. 1.000

In [11]:
# ------------------------------------------------------------------
# 2. Parametri dell’esperimento
# ------------------------------------------------------------------
feature_list   = [5, 50, 100, 300, 400]
correlations   = [0.0, 0.3, 0.7]
n_reps         = 5          # dataset diversi per stimare varianza
n_samples      = 1000
n_classes      = 2
train_size      = 0.70

# dizionario: results[corr][n_features] -> lista accuracy (len = n_reps)
results = {c: {f: [] for f in feature_list} for c in correlations}

# Cartella output
out_dir = Path("output_lda_boost")
out_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# 3. Loop principale: generazione, training, valutazione
# ------------------------------------------------------------------
for corr in correlations:
    for n_feat in feature_list:
        for rep in range(1, n_reps + 1):
            print(f"features: {n_feat}, stage: {rep}, corr: {corr}")
            Σ = generate_cov_matrix(n_feat, mean=corr,
                                    seed=rep, std_dev=0.05)

            X, y = make_classification_cov(n_samples=n_samples,
                                           n_features=n_feat,
                                           n_classes=n_classes,
                                           cov=Σ,
                                           random_state=rep)
            sss = StratifiedShuffleSplit(n_splits=1, train_size=train_size, random_state=1)
            train_idx, test_idx = next(sss.split(X, y))
            X_tr, X_te = X[train_idx], X[test_idx]
            y_tr, y_te = y[train_idx], y[test_idx]

            model = LdaBoost(n_estimators=186,
                             learning_rate=0.05577840617948812,
                             max_depth=3,
                             min_samples_leaf=89,
                             ccp_alpha = 0.00015696331582345908,
                             subsample = 0.7651912630747271,
                             random_state=1)
            model.fit(X_tr, y_tr)
            acc = accuracy_score(y_te, model.predict(X_te))

            results[corr][n_feat].append(acc)
# ------------------------------------------------------------------
# 4. Preparazione statistiche (media ± std) e stampa
# ------------------------------------------------------------------
means = {c: [np.mean(results[c][f]) for f in feature_list]
         for c in correlations}
stds  = {c: [np.std (results[c][f]) for f in feature_list]
         for c in correlations}

print("\n=== Risultati (accuracy media ± std) su", n_reps, "repliche ===")
for c in correlations:
    for idx, f in enumerate(feature_list):
        mu = means[c][idx]
        sd = stds[c][idx]
        print(f"corr={c:>3}, n_features={f:>3} -> mean={mu:.4f}, std={sd:.4f}")

# ------------------------------------------------------------------
# 5. Salvataggio su file (CSV + JSON, incluse le rep grezze)
# ------------------------------------------------------------------
# CSV 'summary'
csv_path = out_dir / "summary_means_stds_binary_1k.csv"
with csv_path.open("w", newline="") as fh:
    writer = csv.writer(fh)
    writer.writerow(["correlation", "n_features", "n_reps", "mean_acc", "std_acc"])
    for c in correlations:
        for idx, f in enumerate(feature_list):
            writer.writerow([c, f, n_reps, f"{means[c][idx]:.6f}", f"{stds[c][idx]:.6f}"])

# JSON 'summary' strutturato per facile riuso
json_summary_path = out_dir / "summary_means_stds_binary_1k.json"
summary_json = {
    "feature_list": feature_list,
    "correlations": correlations,
    "n_reps": n_reps,
    "means": {str(c): means[c] for c in correlations},
    "stds":  {str(c): stds[c]  for c in correlations},
}
with json_summary_path.open("w", encoding="utf-8") as fh:
    json.dump(summary_json, fh, ensure_ascii=False, indent=2)

# JSON con risultati grezzi (per tracciabilità)
json_raw_path = out_dir / "raw_results_binary_1k.json"
raw_json = {str(c): {str(f): results[c][f] for f in feature_list} for c in correlations}
with json_raw_path.open("w", encoding="utf-8") as fh:
    json.dump(raw_json, fh, ensure_ascii=False, indent=2)

print(f"\nFile salvati in: {out_dir.resolve()}")
print(f"- {csv_path.name}")
print(f"- {json_summary_path.name}")
print(f"- {json_raw_path.name}")

features: 5, stage: 1, corr: 0.0
features: 5, stage: 2, corr: 0.0
features: 5, stage: 3, corr: 0.0
features: 5, stage: 4, corr: 0.0
features: 5, stage: 5, corr: 0.0
features: 50, stage: 1, corr: 0.0
features: 50, stage: 2, corr: 0.0
features: 50, stage: 3, corr: 0.0
features: 50, stage: 4, corr: 0.0
features: 50, stage: 5, corr: 0.0
features: 100, stage: 1, corr: 0.0
features: 100, stage: 2, corr: 0.0
features: 100, stage: 3, corr: 0.0
features: 100, stage: 4, corr: 0.0
features: 100, stage: 5, corr: 0.0
features: 300, stage: 1, corr: 0.0
features: 300, stage: 2, corr: 0.0
features: 300, stage: 3, corr: 0.0
features: 300, stage: 4, corr: 0.0
features: 300, stage: 5, corr: 0.0
features: 400, stage: 1, corr: 0.0
features: 400, stage: 2, corr: 0.0
features: 400, stage: 3, corr: 0.0
features: 400, stage: 4, corr: 0.0
features: 400, stage: 5, corr: 0.0
features: 5, stage: 1, corr: 0.3
features: 5, stage: 2, corr: 0.3
features: 5, stage: 3, corr: 0.3
features: 5, stage: 4, corr: 0.3
features

# Three class target, obs. 1.000

In [ ]:
# ------------------------------------------------------------------
# 2. Parametri dell’esperimento
# ------------------------------------------------------------------
feature_list   = [5, 50, 100, 300, 400]
correlations   = [0.0, 0.3, 0.7]
n_reps         = 5          # dataset diversi per stimare varianza
n_samples      = 1000
n_classes      = 3
train_size      = 0.70

# dizionario: results[corr][n_features] -> lista accuracy (len = n_reps)
results = {c: {f: [] for f in feature_list} for c in correlations}

# Cartella output
out_dir = Path("output_lda_boost_three_class")
out_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# 3. Loop principale: generazione, training, valutazione
# ------------------------------------------------------------------
for corr in correlations:
    for n_feat in feature_list:
        for rep in range(1, n_reps + 1):
            print(f"features: {n_feat}, stage: {rep}, corr: {corr}")
            Σ = generate_cov_matrix(n_feat, mean=corr,
                                    seed=rep, std_dev=0.05)

            X, y = make_classification_cov(n_samples=n_samples,
                                           n_features=n_feat,
                                           n_classes=n_classes,
                                           cov=Σ,
                                           random_state=rep)
            sss = StratifiedShuffleSplit(n_splits=1, train_size=train_size, random_state=1)
            train_idx, test_idx = next(sss.split(X, y))
            X_tr, X_te = X[train_idx], X[test_idx]
            y_tr, y_te = y[train_idx], y[test_idx]

            model = LdaBoost(n_estimators=186,
                             learning_rate=0.05577840617948812,
                             max_depth=3,
                             min_samples_leaf=89,
                             ccp_alpha = 0.00015696331582345908,
                             subsample = 0.7651912630747271,
                             random_state=1)
            model.fit(X_tr, y_tr)
            acc = accuracy_score(y_te, model.predict(X_te))

            results[corr][n_feat].append(acc)
# ------------------------------------------------------------------
# 4. Preparazione statistiche (media ± std) e stampa
# ------------------------------------------------------------------
means = {c: [np.mean(results[c][f]) for f in feature_list]
         for c in correlations}
stds  = {c: [np.std (results[c][f]) for f in feature_list]
         for c in correlations}

print("\n=== Risultati (accuracy media ± std) su", n_reps, "repliche ===")
for c in correlations:
    for idx, f in enumerate(feature_list):
        mu = means[c][idx]
        sd = stds[c][idx]
        print(f"corr={c:>3}, n_features={f:>3} -> mean={mu:.4f}, std={sd:.4f}")

# ------------------------------------------------------------------
# 5. Salvataggio su file (CSV + JSON, incluse le rep grezze)
# ------------------------------------------------------------------
# CSV 'summary'
csv_path = out_dir / "summary_means_stds_three_1k.csv"
with csv_path.open("w", newline="") as fh:
    writer = csv.writer(fh)
    writer.writerow(["correlation", "n_features", "n_reps", "mean_acc", "std_acc"])
    for c in correlations:
        for idx, f in enumerate(feature_list):
            writer.writerow([c, f, n_reps, f"{means[c][idx]:.6f}", f"{stds[c][idx]:.6f}"])

# JSON 'summary' strutturato per facile riuso
json_summary_path = out_dir / "summary_means_stds_three_1k.json"
summary_json = {
    "feature_list": feature_list,
    "correlations": correlations,
    "n_reps": n_reps,
    "means": {str(c): means[c] for c in correlations},
    "stds":  {str(c): stds[c]  for c in correlations},
}
with json_summary_path.open("w", encoding="utf-8") as fh:
    json.dump(summary_json, fh, ensure_ascii=False, indent=2)

# JSON con risultati grezzi (per tracciabilità)
json_raw_path = out_dir / "raw_results_three_1k.json"
raw_json = {str(c): {str(f): results[c][f] for f in feature_list} for c in correlations}
with json_raw_path.open("w", encoding="utf-8") as fh:
    json.dump(raw_json, fh, ensure_ascii=False, indent=2)

print(f"\nFile salvati in: {out_dir.resolve()}")
print(f"- {csv_path.name}")
print(f"- {json_summary_path.name}")
print(f"- {json_raw_path.name}")

features: 5, stage: 1, corr: 0.0
features: 5, stage: 2, corr: 0.0
features: 5, stage: 3, corr: 0.0
features: 5, stage: 4, corr: 0.0
features: 5, stage: 5, corr: 0.0
features: 50, stage: 1, corr: 0.0
features: 50, stage: 2, corr: 0.0
features: 50, stage: 3, corr: 0.0
features: 50, stage: 4, corr: 0.0
features: 50, stage: 5, corr: 0.0
features: 100, stage: 1, corr: 0.0
features: 100, stage: 2, corr: 0.0
features: 100, stage: 3, corr: 0.0
features: 100, stage: 4, corr: 0.0
features: 100, stage: 5, corr: 0.0
features: 300, stage: 1, corr: 0.0
features: 300, stage: 2, corr: 0.0
features: 300, stage: 3, corr: 0.0
features: 300, stage: 4, corr: 0.0
features: 300, stage: 5, corr: 0.0
features: 400, stage: 1, corr: 0.0
features: 400, stage: 2, corr: 0.0
features: 400, stage: 3, corr: 0.0
features: 400, stage: 4, corr: 0.0
features: 400, stage: 5, corr: 0.0
features: 5, stage: 1, corr: 0.3
features: 5, stage: 2, corr: 0.3
features: 5, stage: 3, corr: 0.3
features: 5, stage: 4, corr: 0.3
features

----

# Bayes estimator

In [ ]:
import numpy as np
from scipy.stats import truncnorm
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score
from pathlib import Path
import csv
import warnings

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# ============================================================
# 1) Utilità per matrici di correlazione "ben condizionate"
#    (coerenti con il tuo blocco di codice sopra)
# ============================================================
def make_corr_well_conditioned(M: np.ndarray, min_eig: float = 1e-2, kappa_max: float = 1e3) -> np.ndarray:
    """
    Rende M una matrice di *correlazione* ben condizionata:
    - simmetrizza e impone diag=1
    - shrink verso I con alpha scelto per autovalore minimo >= min_eig
      e numero di condizione <= kappa_max
    """
    A = (M + M.T) / 2.0
    np.fill_diagonal(A, 1.0)

    w = np.linalg.eigvalsh(A)
    lam_min = float(w.min())
    lam_max = float(w.max())

    alpha_psd = 0.0
    if lam_min < min_eig:
        alpha_psd = (min_eig - lam_min) / (1.0 - lam_min)

    if kappa_max is None:
        alpha_kappa = 0.0
    else:
        # cond((1-α)A + αI) = ((1-α)λ_max + α) / ((1-α)λ_min + α) <= kappa_max
        Acoef = 1.0 - lam_max - kappa_max * (1.0 - lam_min)
        Bcoef = kappa_max * lam_min - lam_max
        alpha_kappa = float(np.clip(Bcoef / Acoef, 0.0, 0.999))

    alpha = float(np.clip(max(alpha_psd, alpha_kappa, 0.0), 0.0, 0.999))
    R = (1.0 - alpha) * A + alpha * np.eye(A.shape[0])
    R = (R + R.T) / 2.0
    return R


def generate_cov_matrix(size: int, mean: float, seed: int = None,
                        std_dev: float = 0.05,
                        min_eig: float = 1e-2, kappa_max: float = 1e3) -> np.ndarray:
    """
    Matrice di correlazione simmetrica (diag=1) con off-diagonali ~ N(mean, std_dev^2) troncati in [-1,1],
    poi resa ben condizionata.
    """
    if seed is not None:
        np.random.seed(seed)
    a, b = (-1 - mean) / std_dev, (1 - mean) / std_dev
    M = np.eye(size)
    for i in range(size):
        for j in range(i + 1, size):
            v = truncnorm.rvs(a, b, loc=mean, scale=std_dev)
            M[i, j] = M[j, i] = v
    return make_corr_well_conditioned(M, min_eig=min_eig, kappa_max=kappa_max)

# ============================================================
# 2) Generazione dati multiclasse (stesso schema del tuo blocco)
# ============================================================
def class_means(n_features: int, n_classes: int, offset: float = 1.0) -> np.ndarray:
    means = []
    for k in range(n_classes):
        m = np.zeros(n_features)
        for j in range(n_features):
            if (k >> j) & 1:
                m[j] = offset
        means.append(m)
    return np.asarray(means)


def make_classification_cov(n_samples: int, n_features: int, n_classes: int,
                            cov: np.ndarray, random_state: int = None,
                            offset: float = 1.0, shuffle: bool = True):
    rng = np.random.RandomState(random_state)
    Σ = np.array(cov)
    MU = class_means(n_features, n_classes, offset)

    base, extra = divmod(n_samples, n_classes)
    X_parts, y_parts = [], []
    for cls in range(n_classes):
        n = base + (1 if cls < extra else 0)
        X_parts.append(rng.multivariate_normal(MU[cls], Σ, size=n))
        y_parts.append(np.full(n, cls, dtype=int))
    X = np.vstack(X_parts)
    y = np.concatenate(y_parts)

    if shuffle:
        perm = rng.permutation(n_samples)
        X, y = X[perm], y[perm]
    return X, y, MU

# ============================================================
# 3) Classificatore Bayes ottimale (cov comune, priori uguali)
# ============================================================
def bayes_predict_equal_cov(X: np.ndarray, means: np.ndarray, cov: np.ndarray) -> np.ndarray:
    inv_cov = np.linalg.inv(cov)
    w = means @ inv_cov                     # (K × d)
    b = -0.5 * np.sum(w * means, axis=1)    # (K,)
    scores = X @ w.T + b                    # (n × K)
    return np.argmax(scores, axis=1)

# ============================================================
# 4) Esperimento completo + salvataggio CSV
# ============================================================
if __name__ == "__main__":
    # Parametri richiesti
    feature_list     = [5, 50, 100, 300, 400]
    correlations     = [0.0, 0.3, 0.7]
    class_list       = [3]
    n_samples_list   = [1000]
    # class_list       = [2, 5]
    # n_samples_list   = [1000, 2585, 10000]
    n_reps           = 5
    train_fraction   = 0.70  # split 70/30 come richiesto
    rng_split_state  = 1     # per riproducibilità della partizione

    out_dir = Path("output_bayes_optimal")
    out_dir.mkdir(parents=True, exist_ok=True)
    csv_path = out_dir / "bayes_optimal_summary_three_class.csv"

    rows = []
    for n_classes in class_list:
        for n_samples in n_samples_list:
            for corr in correlations:
                for n_feat in feature_list:
                    accs = []
                    for rep in range(1, n_reps + 1):
                        # 1) Crea Σ e dataset con le *stesse* funzioni di campionamento
                        Σ = generate_cov_matrix(n_feat, mean=corr, seed=rep, std_dev=0.05,
                                                min_eig=1e-2, kappa_max=1e3)
                        X, y, MU = make_classification_cov(n_samples=n_samples,
                                                           n_features=n_feat,
                                                           n_classes=n_classes,
                                                           cov=Σ, random_state=rep,
                                                           offset=1.0, shuffle=True)
                        # 2) Split 70/30 (train non usato, si valuta su test)
                        sss = StratifiedShuffleSplit(n_splits=1, train_size=train_fraction, random_state=rng_split_state)
                        tr_idx, te_idx = next(sss.split(X, y))
                        X_te, y_te = X[te_idx], y[te_idx]

                        # 3) Predizione Bayes ottimale sui dati di test
                        y_pred = bayes_predict_equal_cov(X_te, MU, Σ)
                        accs.append(accuracy_score(y_te, y_pred))
                        print(f"Feature: {n_feat}, Correlations: {corr}, Rep: {rep}")

                    mean_acc = float(np.mean(accs))
                    std_acc  = float(np.std(accs))
                    max_acc  = float(np.max(accs))
                    rows.append([n_samples, n_classes, corr, n_feat, n_reps, train_fraction, f"{mean_acc:.3f}", f"{std_acc:.3f}", max_acc])

    # Salva SOLO CSV (niente JSON)
    with csv_path.open("w", newline="") as fh:
        writer = csv.writer(fh)
        writer.writerow(["n_samples", "n_classes", "correlation", "n_features", "n_reps", "train_fraction", "mean_acc", "std_acc", "max_acc"])
        writer.writerows(rows)

    print(f"File CSV salvato in: {csv_path.resolve()}")